In [1]:
import subprocess
def adb(command: str, device_id: str = None):
    """Chạy lệnh ADB, có thể chỉ định thiết bị cụ thể"""
    
    # Nếu có device_id thì thêm -s vào
    if device_id:
        cmd = f"adb -s {device_id} {command}"
    else:
        cmd = f"adb {command}"

    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(result.stdout)
    if result.stderr:
        print("Lỗi:", result.stderr)
    return result.stdout

In [2]:
def get_devices() -> list:
    """Lấy danh sách tất cả thiết bị đang kết nối"""
    result = subprocess.run("adb devices", shell=True, capture_output=True, text=True)
    
    lines = result.stdout.strip().splitlines()
    devices = []
    
    for line in lines[1:]:  # Bỏ dòng "List of devices attached"
        if line.strip() and "device" in line:
            device_id = line.split()[0]
            devices.append(device_id)
    
    return devices


In [3]:
def uninstall_app(package_name: str, device_id: str = None):
    """Gỡ app theo package name"""
    
    print(f"[*] Đang gỡ: {package_name}")
    output = adb(f"uninstall {package_name}", device_id)
    
    if "Success" in output:
        print("[✓] Gỡ thành công!")
    else:
        print("[✗] Gỡ thất bại!")

In [4]:
get_devices()

['192.168.1.19:5555',
 '192.168.1.20:5555',
 '192.168.1.21:5555',
 '192.168.1.22:5555',
 '192.168.1.23:5555',
 '192.168.1.24:5555',
 '192.168.1.25:5555',
 '192.168.1.26:5555',
 '192.168.1.27:5555',
 '192.168.1.28:5555',
 '192.168.1.29:5555',
 '192.168.1.30:5555',
 '192.168.1.31:5555',
 '192.168.1.32:5555',
 '192.168.1.34:5555',
 '192.168.1.35:5555',
 '192.168.1.37:5555',
 '192.168.1.38:5555']

In [5]:
devices = get_devices()
print("Danh sách thiết bị:", devices)
# Output ví dụ:
# ['emulator-5554', '192.168.1.100:5555', 'R3CN109ABCD']

# Bước 2: Chọn thiết bị muốn dùng
device_id = devices[1]  # Lấy thiết bị đầu tiên
print(f"Đang dùng: {device_id}")

Danh sách thiết bị: ['192.168.1.19:5555', '192.168.1.20:5555', '192.168.1.21:5555', '192.168.1.22:5555', '192.168.1.23:5555', '192.168.1.24:5555', '192.168.1.25:5555', '192.168.1.26:5555', '192.168.1.27:5555', '192.168.1.28:5555', '192.168.1.29:5555', '192.168.1.30:5555', '192.168.1.31:5555', '192.168.1.32:5555', '192.168.1.34:5555', '192.168.1.35:5555', '192.168.1.37:5555', '192.168.1.38:5555']
Đang dùng: 192.168.1.20:5555


In [12]:
adb("shell getprop ro.product.model", device_id)

SM-G955F



'SM-G955F\n'

In [6]:
def get_package_name(apk_path: str) -> str:
    """Lấy package name từ file APK"""
    result = subprocess.run(
        f"aapt dump badging {apk_path}",
        shell=True, capture_output=True, text=True
    )
    for line in result.stdout.splitlines():
        if line.startswith("package:"):
            # package: name='com.example.app' ...
            package = line.split("name='")[1].split("'")[0]
            return package
    return None

In [7]:
def install_and_open(apk_path: str, device_id: str = None):
    
    # Bước 1: Cài đặt APK
    print(f"[*] Đang cài: {apk_path}")
    adb(f"install -r {apk_path}", device_id)

    # Bước 2: Lấy package name
    package = get_package_name(apk_path)
    print(f"[*] Package name: {package}")

    if not package:
        print("[!] Không lấy được package name")
        return

    # Bước 3: Lấy main activity để mở app
    result = subprocess.run(
        f"aapt dump badging {apk_path}",
        shell=True, capture_output=True, text=True
    )
    activity = None
    for line in result.stdout.splitlines():
        if "launchable-activity" in line:
            activity = line.split("name='")[1].split("'")[0]
            break

    # Bước 4: Mở app
    if activity:
        print(f"[*] Đang mở: {activity}")
        adb(f"shell am start -n {package}/{activity}", device_id)
    else:
        # Fallback nếu không tìm được activity
        print("[*] Mở app bằng monkey...")
        adb(f"shell monkey -p {package} -c android.intent.category.LAUNCHER 1", device_id)

In [18]:
install_and_open("./apk/instagram-lite-503-0-0-8-107.apk", device_id)

[*] Đang cài: ./apk/instagram-lite-503-0-0-8-107.apk
Performing Streamed Install
Success

[*] Package name: com.instagram.lite
[*] Đang mở: com.facebook.lite.MainActivity
Starting: Intent { cmp=com.instagram.lite/com.facebook.lite.MainActivity }



In [12]:
def tap(x: int, y: int, device_id: str = None):
    """Click vào tọa độ x, y"""
    adb(f"shell input tap {x} {y}", device_id)

def swipe(x1: int, y1: int, x2: int, y2: int, duration_ms: int = 300, device_id: str = None):
    """Vuốt từ điểm 1 đến điểm 2"""
    adb(f"shell input swipe {x1} {y1} {x2} {y2} {duration_ms}", device_id)

In [13]:
def input_text_unicode(text: str, device_id: str = None):
    """Nhập text unicode - hỗ trợ tiếng Việt"""
    import base64
    
    # Encode text sang base64
    encoded = base64.b64encode(text.encode('utf-8')).decode('utf-8')
    
    # Cần cài ADBKeyboard trên thiết bị
    adb(f"shell am broadcast -a ADB_INPUT_B64 --es msg '{encoded}'", device_id)

In [30]:
import time
import random

def input_text(text: str, device_id: str = None, delay: float = 0.1):
    """Nhập từng ký tự một, giả lập người thật"""
    
    text = text.replace(" ", "%s")
    
    for char in text:
        adb(f"shell input text '{char}'", device_id)
        
        # Delay ngẫu nhiên quanh mức delay cho tự nhiên hơn
        time.sleep(delay + random.uniform(-0.03, 0.03))

In [14]:
adb("shell settings put system pointer_location 1", device_id)

''

In [23]:
# Allow Access contact
tap(1000, 1600, device_id)

In [24]:
# Click Create new Account
tap(600, 1500, device_id)

In [25]:
# Sign up with email
tap(700, 1025, device_id)

In [31]:
input_text("test@hotmail.com", device_id)

In [33]:
tap(700, 760, device_id)

In [34]:
# sleep 5s
# Get code HERE
otp = '136597'
input_text(otp, device_id)

In [35]:
tap(700, 700, device_id)

In [38]:
fullname = "Nguyen Van A"
password = '123@123@123'
input_text_unicode(fullname, device_id)
tap(700, 600, device_id)
input_text(password, device_id)

Broadcasting: Intent { act=ADB_INPUT_B64 flg=0x400000 (has extras) }
Broadcast completed: result=0















In [39]:
tap(700, 950, device_id)

In [40]:
tap(250, 2500, device_id)

In [41]:
swipe(720, 2500, 720, 2200, 300, device_id)

In [42]:
tap(720, 2500, device_id)

In [ ]:
for i in range(random.randint(10, 15)):
    swipe(1200, 2500, 1200, 2200, 300, device_id)
tap(1200, 2500, device_id)

In [ ]:
# Register
tap(700, 2700, device_id)

In [46]:
# your account is almost ready -> Next
tap(700, 2600, device_id)

In [8]:
from src.lib import *

In [9]:
devices = get_devices()
phone = PhoneDevice(devices[1])

In [5]:
phone.swipe(720, 2500, 720, 2200, 300)

In [6]:
phone.proxy_config.set_proxy(
    "117.2.176.90",
    "48561"
)

[✓] Proxy set: 117.2.176.90:48561


In [10]:
get_package_name("./apk/instagram-lite-503-0-0-8-107.apk")

'com.instagram.lite'

In [11]:
phone.get_cookie("com.instagram.lite")

[*] Copying cookies DB from com.instagram.lite...
Lỗi: /system/bin/sh: no closing quote

adb: error: failed to stat remote object '/sdcard/_cookies_tmp.db': No such file or directory

[!] Không lấy được file cookies.db — kiểm tra thiết bị đã root chưa.


''

In [14]:
import subprocess
import os
import sys

PACKAGE = "com.instagram.lite"
SDCARD_PATH = "/sdcard/insta_lite_dump"
LOCAL_DUMP = "insta_lite_dump"

def run_adb(cmd: str, shell=False):
    try:
        result = subprocess.run(cmd, shell=shell, capture_output=True, text=True, check=True)
        return result.stdout.strip()
    except subprocess.CalledProcessError as e:
        print(f"❌ Lỗi: {e.stderr}")
        return None

print("🔄 Force stop Instagram Lite...")
run_adb(f"adb -s {device_id} shell am force-stop {PACKAGE}")

print("🔍 Tìm tất cả file cookie/database trong package...")
find_cmd = f'adb -s {device_id} shell "su -c \'find /data/data/{PACKAGE} -type f \\( -name "*cookie*" -o -name "*.db" -o -name "shared_prefs*" \\) 2>/dev/null | head -30\'"'
files = run_adb(find_cmd, shell=True)

if not files:
    print("❌ Không tìm thấy file nào. App có thể dùng encrypted storage.")
    print("   Thử mở app → xem vài reel/story/profile để tạo dữ liệu rồi chạy lại.")
    sys.exit(1)

print("📁 Các file tìm thấy:\n")
print(files)

# Tạo thư mục dump
run_adb(f'adb -s {device_id} shell "su -c \'mkdir -p {SDCARD_PATH} && chmod 755 {SDCARD_PATH}\'"', shell=True)

print("\n📦 Copy tất cả file quan trọng ra /sdcard...")
copy_cmd = f'adb -s {device_id} shell "su -c \'cp -r /data/data/{PACKAGE}/shared_prefs {SDCARD_PATH}/ 2>/dev/null; '
copy_cmd += f'find /data/data/{PACKAGE} -name "*.db" -exec cp {{}} {SDCARD_PATH}/ \\; 2>/dev/null; '
copy_cmd += f'find /data/data/{PACKAGE} -name "*cookie*" -exec cp {{}} {SDCARD_PATH}/ \\; \' "'
run_adb(copy_cmd, shell=True)

print("📥 Pull toàn bộ dump về máy...")
run_adb(f"adb -s {device_id} pull {SDCARD_PATH} {LOCAL_DUMP}", shell=True)

print(f"\n✅ Dump hoàn tất! Kiểm tra thư mục: **{LOCAL_DUMP}**")
print("   - Mở các file .xml trong shared_prefs để tìm token/session")
print("   - Dùng DB Browser for SQLite mở các file .db")
print("   - Tìm từ khóa: sessionid, csrftoken, ds_user_id, mid, ig_did, access_token")

# Gợi ý tìm nhanh trong shared_prefs
print("\n🔎 Kiểm tra nhanh shared_prefs:")
run_adb(f'adb -s {device_id} shell "su -c \'grep -E "session|token|user_id" /data/data/{PACKAGE}/shared_prefs/*.xml 2>/dev/null || echo "Không tìm thấy trong xml"\'"', shell=True)

🔄 Force stop Instagram Lite...
🔍 Tìm tất cả file cookie/database trong package...
❌ Không tìm thấy file nào. App có thể dùng encrypted storage.
   Thử mở app → xem vài reel/story/profile để tạo dữ liệu rồi chạy lại.


SystemExit: 1

c:\Program Files\Python311\Lib\site-packages\IPython\core\interactiveshell.py:3516: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
